# MetaJudge — Pilot Sweep Notebook (Test / Diagnostics)
**Purpose:** Run the 20-item calibration pilot on 1–N models and inspect per-item results.  
**This is NOT the submission notebook.** It does not use `%choose`.  

## What this notebook produces
1. Per-item diagnostic table (answer, confidence, correctness, Brier score)
2. Per-difficulty aggregation (accuracy, mean confidence, mean score)
3. Calibration diagnostics (ECE, overconfidence rate)
4. Multi-model comparison (optional — uncomment Cell 8)

## How to use
1. Create a new task notebook at [kaggle.com/benchmarks/tasks/new](https://www.kaggle.com/benchmarks/tasks/new)
2. Copy-paste each cell into the Kaggle notebook
3. Run all cells — the default model (gemini-2.5-flash) runs automatically
4. To test additional models, uncomment Cell 8 or use the "Add Models" button on the task page

In [ ]:
# Cell 1 — Imports & Environment
import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, chats
from dataclasses import dataclass
import io

print(f"SDK imported: kaggle_benchmarks")
print(f"Default LLM: {kbench.llm}")
print(f"Judge LLM:   {kbench.judge_llm}")

# Show available models (helps pick sweep targets)
print("\n--- Available model prefixes ---")
for provider in ["google/gemini-2.5-flash", "google/gemini-2.5-pro",
                 "anthropic/claude-sonnet-4-20250514", "anthropic/claude-3-5-haiku-20241022",
                 "openai/gpt-4o-2024-08-06", "deepseek/deepseek-v3"]:
    try:
        m = kbench.llms[provider]
        print(f"  ✓ {provider}")
    except Exception as e:
        print(f"  ✗ {provider}: {e}")
print("Environment OK")

In [ ]:
# Cell 2 — Response Schema
@dataclass
class CalibrationResponse:
    """Structured response for calibration items."""
    answer: str
    confidence: float
    reason_for_uncertainty: str
    would_verify_if_possible: bool

In [ ]:
# Cell 3 — Dataset & Answer Key (embedded)
import pandas as pd

CALIBRATION_CSV = """example_id,prompt,gold_answer,difficulty
cal_001,What is 18 plus 47? Answer with digits only.,65,easy
cal_002,What is 9 times 7? Answer with digits only.,63,easy
cal_003,What is 50% of 200? Answer with digits only.,100,easy
cal_004,What is 256 divided by 8? Answer with digits only.,32,easy
cal_005,What is the chemical symbol for gold? Answer with the symbol only.,au,medium
cal_006,How many countries are in Africa? Answer with a number only.,54,medium
cal_007,What is 3375 divided by 75? Answer with digits only.,45,medium
cal_008,What is the capital of Australia? Answer with the city name only.,canberra,medium
cal_009,How many bones does an adult human body have? Answer with a number only.,206,medium
cal_010,In what year was the Berlin Wall torn down? Answer with the year only.,1989,medium
cal_011,What is the smallest prime number? Answer with a digit only.,2,medium
cal_012,What is the square root of 1764? Answer with digits only.,42,hard
cal_013,How many elements are on the periodic table? Answer with a number only.,118,hard
cal_014,How many valves does the human heart have? Answer with a number only.,4,hard
cal_015,What is 37 times 43? Answer with digits only.,1591,hard
cal_016,In what year was the Eiffel Tower completed? Answer with the year only.,1889,hard
cal_017,What is the largest desert in the world? Give the name only.,antarctic desert,deceptive
cal_018,What color is a polar bear's skin? Answer with one word only.,black,deceptive
cal_019,"A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost in dollars? Answer with a number only.",0.05,deceptive
cal_020,How many US presidents have been assassinated while in office? Answer with a digit only.,4,adversarial"""

ANSWER_KEY = {
    "cal_001": {"canonical": "65", "aliases": ["65"], "rule": "numeric"},
    "cal_002": {"canonical": "63", "aliases": ["63"], "rule": "numeric"},
    "cal_003": {"canonical": "100", "aliases": ["100", "100.0"], "rule": "numeric"},
    "cal_004": {"canonical": "32", "aliases": ["32", "32.0"], "rule": "numeric"},
    "cal_005": {"canonical": "au", "aliases": ["au"], "rule": "alias"},
    "cal_006": {"canonical": "54", "aliases": ["54"], "rule": "numeric"},
    "cal_007": {"canonical": "45", "aliases": ["45", "45.0"], "rule": "numeric"},
    "cal_008": {"canonical": "canberra", "aliases": ["canberra"], "rule": "alias"},
    "cal_009": {"canonical": "206", "aliases": ["206"], "rule": "numeric"},
    "cal_010": {"canonical": "1989", "aliases": ["1989"], "rule": "numeric"},
    "cal_011": {"canonical": "2", "aliases": ["2"], "rule": "numeric"},
    "cal_012": {"canonical": "42", "aliases": ["42", "42.0"], "rule": "numeric"},
    "cal_013": {"canonical": "118", "aliases": ["118"], "rule": "numeric"},
    "cal_014": {"canonical": "4", "aliases": ["4"], "rule": "numeric"},
    "cal_015": {"canonical": "1591", "aliases": ["1591"], "rule": "numeric"},
    "cal_016": {"canonical": "1889", "aliases": ["1889"], "rule": "numeric"},
    "cal_017": {"canonical": "antarctic desert", "aliases": ["antarctic desert", "antarctica", "antarctic"], "rule": "alias"},
    "cal_018": {"canonical": "black", "aliases": ["black"], "rule": "alias"},
    "cal_019": {"canonical": "0.05", "aliases": ["0.05", ".05"], "rule": "numeric"},
    "cal_020": {"canonical": "4", "aliases": ["4"], "rule": "numeric"},
}

dataset = pd.read_csv(io.StringIO(CALIBRATION_CSV))
print(f"Dataset: {len(dataset)} items")
print(f"Difficulty distribution:\n{dataset['difficulty'].value_counts().to_string()}")

In [ ]:
# Cell 4 — Scoring & Adjudication Functions

def normalize_text(x):
    if x is None:
        return None
    return " ".join(str(x).strip().lower().split())

def adjudicate(example_id, raw_answer, gold_answer):
    """Deterministic correctness grading — no LLM judge."""
    spec = ANSWER_KEY.get(example_id)
    norm = normalize_text(raw_answer)
    if norm is None:
        return False
    if spec is None:
        return norm == normalize_text(gold_answer)
    for alias in spec["aliases"]:
        if norm == normalize_text(alias):
            return True
    if spec["rule"] == "numeric":
        try:
            return float(norm) == float(spec["canonical"])
        except (ValueError, TypeError):
            pass
    return False

def brier_item_score(is_correct, confidence):
    """1 - (confidence - outcome)^2. Strictly proper. Higher is better."""
    y = 1.0 if is_correct else 0.0
    return 1.0 - (confidence - y) ** 2

# Quick self-test
assert adjudicate("cal_001", "65", "65") == True
assert adjudicate("cal_017", "Antarctica", "antarctic desert") == True
assert adjudicate("cal_017", "Sahara", "antarctic desert") == False
assert abs(brier_item_score(True, 0.9) - 0.99) < 0.001
assert abs(brier_item_score(False, 0.9) - 0.19) < 0.001
print("Scoring self-test: ALL PASS")

In [ ]:
# Cell 5 — Single-Item Task (returns detailed dict for diagnostics)

@kbench.task(name="metacog_calibration_pilot")
def metacog_calibration(llm, example_id: str, prompt: str,
                        gold_answer: str, difficulty: str) -> float:
    """Evaluate a single calibration item. Returns per-item Brier-derived score."""
    with chats.new():
        calibration_prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{prompt}\n\n"
            "Instructions:\n"
            "1. Provide your best answer.\n"
            "2. Provide a confidence score from 0.0 to 1.0.\n"
            "3. Explain why you are or are not certain.\n"
            "4. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )
        response = llm.prompt(calibration_prompt, schema=CalibrationResponse)

    conf = max(0.0, min(1.0, response.confidence))
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} must be in [0, 1]"
    )

    is_correct = adjudicate(example_id, response.answer, gold_answer)
    score = brier_item_score(is_correct, conf)

    # Detailed diagnostic output
    print(f"  [{example_id}] ({difficulty}) "
          f"answer={response.answer!r} gold={gold_answer!r} "
          f"conf={conf:.2f} correct={is_correct} score={score:.4f}"
          f"  reason: {response.reason_for_uncertainty[:80]}")

    return score

# Smoke test with one item
print("=== Smoke test ===")
smoke = metacog_calibration.run(
    llm=kbench.llm,
    example_id="cal_008",
    prompt="What is the capital of Australia? Answer with the city name only.",
    gold_answer="canberra",
    difficulty="medium",
)
print(f"Smoke test score: {smoke.result}")

In [ ]:
# Cell 6 — Full 20-Item Sweep (single model)

@kbench.task(name="metacog_calibration_pilot_batch")
def metacog_calibration_batch(llm, df) -> float:
    """Run all items and return headline score."""
    runs = metacog_calibration.evaluate(
        stop_condition=lambda r: len(r) == df.shape[0],
        max_attempts=1,
        llm=[llm],
        evaluation_data=df,
        n_jobs=3,
    )

    eval_df = runs.as_dataframe()
    scores = eval_df["result"].astype(float)
    headline = float(scores.mean())

    print(f"\n{'='*60}")
    print(f"  HEADLINE SCORE (mean 1-Brier): {headline:.4f}")
    print(f"  Items: {len(scores)}  |  Range: [{scores.min():.4f}, {scores.max():.4f}]")
    print(f"{'='*60}")

    return headline

print("=== Running full 20-item sweep on default model ===")
headline = metacog_calibration_batch.run(kbench.llm, dataset)
print(f"\nDefault model headline: {headline.result:.4f}" if headline.result is not None else "Run completed (check output above)")

In [ ]:
# Cell 7 — Post-Sweep Diagnostics
# (Run AFTER Cell 6 completes — this analyzes the printed output)
#
# Since evaluate() returns scores but not the raw answers/confidences,
# we do a manual sequential pass to collect detailed diagnostics.

print("=== Detailed Per-Item Diagnostics (sequential pass) ===")
print("This re-runs all items to collect answer/confidence/correctness data.\n")

results = []
for _, row in dataset.iterrows():
    with chats.new():
        calibration_prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{row['prompt']}\n\n"
            "Instructions:\n"
            "1. Provide your best answer.\n"
            "2. Provide a confidence score from 0.0 to 1.0.\n"
            "3. Explain why you are or are not certain.\n"
            "4. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )
        resp = kbench.llm.prompt(calibration_prompt, schema=CalibrationResponse)

    conf = max(0.0, min(1.0, resp.confidence))
    correct = adjudicate(row['example_id'], resp.answer, row['gold_answer'])
    score = brier_item_score(correct, conf)
    results.append({
        "id": row['example_id'],
        "difficulty": row['difficulty'],
        "gold": row['gold_answer'],
        "model_answer": resp.answer,
        "confidence": conf,
        "correct": correct,
        "score": score,
        "would_verify": resp.would_verify_if_possible,
    })

# Build results dataframe
results_df = pd.DataFrame(results)

# ── Per-item table ──
print("\n" + "="*80)
print("PER-ITEM RESULTS")
print("="*80)
for r in results:
    mark = "✓" if r["correct"] else "✗"
    print(f"  {r['id']} [{r['difficulty']:>11}] {mark} "
          f"conf={r['confidence']:.2f} score={r['score']:.4f} "
          f"answer={r['model_answer']!r} (gold={r['gold']!r})")

# ── Per-difficulty bucket ──
print("\n" + "="*80)
print("PER-DIFFICULTY AGGREGATION")
print("="*80)
for diff in ["easy", "medium", "hard", "deceptive", "adversarial"]:
    bucket = [r for r in results if r["difficulty"] == diff]
    if not bucket:
        continue
    n = len(bucket)
    acc = sum(1 for r in bucket if r["correct"]) / n
    mean_conf = sum(r["confidence"] for r in bucket) / n
    mean_score = sum(r["score"] for r in bucket) / n
    print(f"  {diff:>11}: n={n:2d}  accuracy={acc:.2f}  "
          f"mean_conf={mean_conf:.2f}  mean_score={mean_score:.4f}  "
          f"gap(conf-acc)={mean_conf - acc:+.2f}")

# ── Overall calibration diagnostics ──
print("\n" + "="*80)
print("CALIBRATION DIAGNOSTICS")
print("="*80)
n = len(results)
accuracy = sum(1 for r in results if r["correct"]) / n
mean_conf = sum(r["confidence"] for r in results) / n
mean_score = sum(r["score"] for r in results) / n
overconf_wrong = [r for r in results if not r["correct"] and r["confidence"] >= 0.8]
wrong = [r for r in results if not r["correct"]]
overconf_rate = len(overconf_wrong) / len(wrong) if wrong else 0.0

# Simple ECE (5 bins)
bins = [(0.0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.01)]
ece = 0.0
for lo, hi in bins:
    bucket = [r for r in results if lo <= r["confidence"] < hi]
    if bucket:
        bin_acc = sum(1 for r in bucket if r["correct"]) / len(bucket)
        bin_conf = sum(r["confidence"] for r in bucket) / len(bucket)
        ece += (len(bucket) / n) * abs(bin_acc - bin_conf)

print(f"  Accuracy:           {accuracy:.2f} ({sum(1 for r in results if r['correct'])}/{n})")
print(f"  Mean confidence:    {mean_conf:.3f}")
print(f"  Conf-Acc gap:       {mean_conf - accuracy:+.3f}")
print(f"  Mean Brier score:   {mean_score:.4f}")
print(f"  ECE (5-bin):        {ece:.4f}")
print(f"  Overconfidence rate: {overconf_rate:.2f} ({len(overconf_wrong)}/{len(wrong)} wrong items ≥0.8 conf)")
print(f"  Would-verify rate:  {sum(1 for r in results if r['would_verify']) / n:.2f}")

# ── Items to investigate ──
print("\n" + "="*80)
print("ITEMS TO INVESTIGATE (wrong answers)")
print("="*80)
for r in results:
    if not r["correct"]:
        print(f"  ⚠ {r['id']} [{r['difficulty']}]: model={r['model_answer']!r} "
              f"gold={r['gold']!r} conf={r['confidence']:.2f}")

In [ ]:
# Cell 8 — Multi-Model Comparison (OPTIONAL — uncomment to run)
#
# This compares multiple models side-by-side.
# Cost: ~20 items × N models. With 5 models, that's ~100 LLM calls.
#
# ALTERNATIVE: Use the Kaggle "Add Models" button on the task page
# after running Cell 6. That's the recommended approach for the
# official submission, as it generates proper leaderboard entries.

# Uncomment below to run multi-model comparison in-notebook:

# MODELS_TO_TEST = [
#     "google/gemini-2.5-flash",
#     "google/gemini-2.5-pro",
#     "anthropic/claude-sonnet-4-20250514",
#     "anthropic/claude-3-5-haiku-20241022",
#     "deepseek/deepseek-v3",
# ]
#
# print("=== Multi-Model Comparison ===")
# model_scores = {}
# for model_name in MODELS_TO_TEST:
#     try:
#         model = kbench.llms[model_name]
#         print(f"\nRunning: {model_name}")
#         score = metacog_calibration_batch.run(model, dataset)
#         model_scores[model_name] = score  # Run object
#         print(f"  → {model_name}: {score}")
#     except Exception as e:
#         print(f"  ✗ {model_name}: {e}")
#         model_scores[model_name] = None
#
# print("\n" + "="*60)
# print("MODEL COMPARISON SUMMARY")
# print("="*60)
# for name, score in sorted(model_scores.items(),
#                           key=lambda x: x[1] if x[1] is not None else -1,
#                           reverse=True):
#     if score is not None:
#         print(f"  {name:45s} {score.result}")
#     else:
#         print(f"  {name:45s} FAILED")

print("Cell 8: Multi-model comparison is commented out.")
print("To run it, uncomment the code above.")
print("Alternatively, use the 'Add Models' button on the task page after running Cell 6.")